In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import google.auth
import os
import gcsfs
import requests
import fsspec
from shapely import wkt
import re
from calitp_data_analysis.sql import get_engine
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()


pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
# Load the stored ACS dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/census_tracts_data.parquet", "rb") as f:
    tracts_ca_acs = gpd.read_parquet(f)

In [5]:
# Load the stored organization, ridership, stop, data from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/ridership_trips_routes_weekday.csv", "rb") as f:
    ridership_trips_routes_weekday = pd.read_csv(f)

In [6]:
# Load job density data from GCS and select required columns
# Open the GCS file using your existing fs object
with fs.open(f"{GCS_FILE_PATH}/job_density_2022.parquet", "rb") as f:
    jobdata = pd.read_parquet(f)

# Select only the columns you want, including geometry
jobdata = jobdata[['GEOID', 'jobs_tot']]

## Spatial Analysis: Stop Buffers and Census Tract Intersections

In [7]:
ridership_trips_routes_weekday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28545 entries, 0 to 28544
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   organization_name         28545 non-null  object 
 1   feed_key                  28544 non-null  object 
 2   route_id                  26875 non-null  object 
 3   route_name                13556 non-null  object 
 4   stop_id                   28544 non-null  object 
 5   stop_name                 28545 non-null  object 
 6   stop_code                 27483 non-null  object 
 7   n_trips                   28544 non-null  float64
 8   n_routes                  28544 non-null  float64
 9   pt_geom                   28544 non-null  object 
 10  day_type                  28545 non-null  object 
 11  average_daily_boardings   28545 non-null  float64
 12  average_daily_alightings  25942 non-null  float64
 13  start_date                28545 non-null  object 
 14  end_da

In [8]:
# Drop rows with missing pt_geom
ridership_trips_routes_weekday = ridership_trips_routes_weekday[
    ridership_trips_routes_weekday['pt_geom'].notna() & 
    (ridership_trips_routes_weekday['pt_geom'] != 'nan')
].copy()


In [9]:
# Ensure pt_geom is string type
ridership_trips_routes_weekday['pt_geom'] = ridership_trips_routes_weekday['pt_geom'].astype(str)

In [10]:
# Convert pt_geom column from WKT to shapely geometry
ridership_trips_routes_weekday['geometry'] = ridership_trips_routes_weekday['pt_geom'].apply(wkt.loads)

# Create a GeoDataFrame
gdf_ridership = gpd.GeoDataFrame(ridership_trips_routes_weekday, geometry='geometry')

In [11]:
# Set CRS (assuming WGS84)
gdf_ridership.set_crs(epsg=4326, inplace=True)

,organization_name,feed_key,route_id,route_name,stop_id,stop_name,stop_code,n_trips,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date,geometry
0,Gold Coast Transit,3cb676436aad669e52042c0e97a9a051,16,NaN,VNACLR1,.,NaN,32.0,1.0,POINT(-119.294028 34.343645),Weekday,0.000000,3.000000,2025-05-01,2025-05-31,POINT (-119.29403 34.34365)
1,Samtrans,db97cc02836aa5f0cf647d80160b23ec,ECR,NaN,345017,1000 El Camino Real-Menlo College,345017,143.0,1.0,POINT(-122.191284 37.457543),Weekday,9.523810,24.571429,2025-08-01,2025-08-31,POINT (-122.19128 37.45754)
2,Golden Gate Transit,de77cb40e92fb47fa8d16228977cfb86,130,NaN,40469,1011 Andersen Dr,40469,39.0,1.0,POINT(-122.504252 37.955391),Weekday,1.909091,0.000000,2025-09-01,2025-09-30,POINT (-122.50425 37.95539)
3,Samtrans,db97cc02836aa5f0cf647d80160b23ec,61,NaN,343119,1011 Crestview Dr,343119,5.0,1.0,POINT(-122.284103 37.484282),Weekday,1.444444,1.888889,2025-08-01,2025-08-31,POINT (-122.28410 37.48428)
4,Samtrans,db97cc02836aa5f0cf647d80160b23ec,46,NaN,340606,1060 Carolan Ave,340606,7.0,1.0,POINT(-122.359627 37.586685),Weekday,10.333333,5.500000,2025-08-01,2025-08-31,POINT (-122.35963 37.58669)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28540,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,NaN,NaN,70091,San Mateo,NaN,104.0,3.0,POINT(-122.323851 37.568087),Weekday,1074.189854,NaN,2023-11-01,2025-07-31,POINT (-122.32385 37.56809)
28541,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,NaN,NaN,70241,Santa Clara,NaN,90.0,2.0,POINT(-121.93608 37.353238),Weekday,718.467622,NaN,2023-11-01,2025-07-31,POINT (-121.93608 37.35324)
28542,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,NaN,NaN,70041,South San Francisco,NaN,104.0,3.0,POINT(-122.404979051 37.655941395),Weekday,559.501648,NaN,2023-11-01,2025-07-31,POINT (-122.40498 37.65594)
28543,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,NaN,NaN,70221,Sunnyvale,NaN,104.0,3.0,POINT(-122.031372 37.378916),Weekday,1408.195136,NaN,2023-11-01,2025-07-31,POINT (-122.03137 37.37892)


In [12]:
# Reproject to match census tracts CRS
gdf_ridership = gdf_ridership.to_crs(tracts_ca_acs.crs)

In [13]:
stop_buffered = gdf_ridership.copy()
stop_buffered["geometry"] = stop_buffered.geometry.buffer(804.672)

In [14]:
# Inner join with ACS data on 'geo_id'
tracts_ca_acs = tracts_ca_acs.merge(jobdata, on = 'GEOID', how='left')

In [15]:
tracts_ca_acs.crs

<Projected CRS: EPSG:3310>
Name: NAD83 / California Albers
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: United States (USA) - California.
- bounds: (-124.45, 32.53, -114.12, 42.01)
Coordinate Operation:
- name: California Albers
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [16]:
stop_buffered.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 28544 entries, 0 to 28544
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   organization_name         28544 non-null  object  
 1   feed_key                  28544 non-null  object  
 2   route_id                  26875 non-null  object  
 3   route_name                13556 non-null  object  
 4   stop_id                   28544 non-null  object  
 5   stop_name                 28544 non-null  object  
 6   stop_code                 27483 non-null  object  
 7   n_trips                   28544 non-null  float64 
 8   n_routes                  28544 non-null  float64 
 9   pt_geom                   28544 non-null  object  
 10  day_type                  28544 non-null  object  
 11  average_daily_boardings   28544 non-null  float64 
 12  average_daily_alightings  25942 non-null  float64 
 13  start_date                28544 non-null  o

In [17]:
geometry_intersect = gpd.overlay(
    stop_buffered, 
    tracts_ca_acs, 
    how='intersection', 
    keep_geom_type=True
)

In [18]:
# Calculate intersected area
geometry_intersect['area_2'] = geometry_intersect.geometry.area

# Calculate the proportion of the tract that intersects each stop
geometry_intersect['area_ratio'] = geometry_intersect['area_2'] / geometry_intersect['area_m2']

In [19]:
# Define demographic and socioeconomic columns to be adjusted by area ratio
cols_to_weight = [
    'total_pop', 'poverty_pop', 'non_us_citizen', 'workers_with_no_car', 
    'households_with_no_cars', 'disabled_pop', 'public_asst_pop', 
    'inc_extremelylow', 'inc_verylow', 'inc_low', 
    'male_seniors', 'female_seniors', 'veteran_pop', 'male_youth',  'female_youth', 'jobs_tot', 'ALAND'
]

# Apply area_ratio
for col in cols_to_weight:
    geometry_intersect[f'{col}_adj'] = geometry_intersect[col] * geometry_intersect['area_ratio']

In [20]:
geometry_intersect.organization_name.unique()

array(['Gold Coast Transit', 'Samtrans', 'Golden Gate Transit', 'SDMTS',
       'SacRT Bus', 'Orange County Transportation Authority',
       'Long Beach Transit', 'Foothill Transit', 'Fresno County',
       'San Francisco Bay Area Rapid Transit District', 'Big Blue Bus',
       'Culver City Bus', 'Riverside Transit', 'Golden Gate Park Shuttle',
       'Caltrain'], dtype=object)

In [21]:
stop_acs_rollup = geometry_intersect.groupby(
    ['feed_key', 'stop_id', 'organization_name', 'geometry'], 
    as_index=False
)[[f'{col}_adj' for col in cols_to_weight]].sum()

In [27]:
stop_route_df = gdf_ridership.merge(
    stop_acs_rollup,
    on=['feed_key', 'stop_id','organization_name'],
    how='left'
)

In [28]:
stop_route_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151044 entries, 0 to 151043
Data columns (total 34 columns):
 #   Column                       Non-Null Count   Dtype   
---  ------                       --------------   -----   
 0   organization_name            151044 non-null  object  
 1   feed_key                     151044 non-null  object  
 2   route_id                     143325 non-null  object  
 3   route_name                   71252 non-null   object  
 4   stop_id                      151044 non-null  object  
 5   stop_name                    151044 non-null  object  
 6   stop_code                    145743 non-null  object  
 7   n_trips                      151044 non-null  float64 
 8   n_routes                     151044 non-null  float64 
 9   pt_geom                      151044 non-null  object  
 10  day_type                     151044 non-null  object  
 11  average_daily_boardings      151044 non-null  float64 
 12  average_daily_alightings     140796 non-null

In [29]:
stop_route_df.organization_name.unique()

array(['Gold Coast Transit', 'Samtrans', 'Golden Gate Transit', 'SDMTS',
       'SacRT Bus', 'Orange County Transportation Authority',
       'Long Beach Transit', 'Foothill Transit', 'Fresno County',
       'San Francisco Bay Area Rapid Transit District', 'Big Blue Bus',
       'Culver City Bus', 'Riverside Transit', 'Golden Gate Park Shuttle',
       'Caltrain'], dtype=object)

In [31]:
stop_route_df = gpd.GeoDataFrame(
    stop_route_df, 
    geometry='geometry_x', 
    crs=geometry_intersect.crs
)

In [32]:
# Store data in warehouse
with fs.open(f"{GCS_FILE_PATH}/census_stop_ridershipdata.parquet", "wb") as f:
    stop_route_df.to_parquet(f, index=False)